# LearnRAG: the complete pipeline, step by step

**Documents → chunks → embeddings → search → evidence in a prompt → generated answer.**

This notebook contains the main implementation directly, so you can explain it without switching to the Python file. The matching script remains available for command-line use and component tests. No paid API key is needed. Initial downloads require Internet and several GB of space; a Colab T4 GPU is recommended.

Run cells in order. The final section can display previously recorded results without loading either model.

## 1. Get the code and install dependencies

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. The next cell clones LearnRAG only if the current folder does not already contain its script. Existing files are reused; it does not pull or overwrite changes.


In [1]:
# Get the project only if it is not already here. Leave any existing files alone.

from pathlib import Path
import os, subprocess

if not Path("simple_rag.py").is_file():
    project = Path.cwd() / "LearnRAG"
    if not project.exists():
        subprocess.run(["git", "clone", "https://github.com/nafismam/LearnRAG.git", str(project)], check=True)
    if not (project / "simple_rag.py").is_file():
        raise RuntimeError("LearnRAG exists but does not contain simple_rag.py; select the correct project directory.")
    os.chdir(project)
print("Project directory:", Path.cwd())


Project directory: /workspace/LearnRAG


In [2]:
# Install the packages. Model files are downloaded later when we load the models.

%pip install -r requirements.txt



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Imports
Python tools for storing settings, reading files, and working with vectors.

In [1]:
from __future__ import annotations

import argparse
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

'false'

## 1. Settings

Keep the model names, chunk sizes and number of retrieved passages in one place. These are demonstration defaults, not settings proven optimal by an experiment.

In [2]:
# SETTINGS: keep the model names and search settings together so they are easy to change.
# These defaults are for a small example; they were not chosen by an accuracy study.
@dataclass
class RAGConfig:

    # Read text and Markdown files from this folder.
    data_dir: str = "data"


    # This model turns English text into 384 numbers used for matching.
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"


    # This is a different model: it writes the answer from the retrieved text.
    llm_model: str = "Qwen/Qwen2.5-1.5B-Instruct"


    # Cut at most 500 characters at a time. Repeat 100 characters across each boundary.
    # This helps keep some context, but can still cut sentences in half.
    chunk_size: int = 500
    chunk_overlap: int = 100


    # Return this many passages, even if their scores are low. No relevance gate here.
    top_k: int = 3


    max_new_tokens: int = 256
    # Zero turns off sampling. It does not make every answer true or fully repeatable everywhere.
    temperature: float = 0.0

## 2. Load the documents

Read text and Markdown files. Keep each file’s relative path with its text so we can show where an answer’s evidence came from.

In [3]:
# 1. LOAD: keep the document text together with the path it came from.
@dataclass
class Document:
    text: str
    source: str


# Read files in a consistent order. Skip empty files and stop if no usable files exist.
def load_documents(data_dir: str) -> list[Document]:
    docs: list[Document] = []
    for path in sorted(Path(data_dir).rglob("*")):

        # Keep the folder in the source name: two folders can both contain a file called note.md.
        if path.suffix.lower() in {".txt", ".md"} and path.is_file():
            text = path.read_text(encoding="utf-8", errors="ignore").strip()
            if text:
                docs.append(Document(text=text, source=path.relative_to(data_dir).as_posix()))
    if not docs:
        raise FileNotFoundError(f"No .txt or .md files found in '{data_dir}'.")
    print(f"[load] {len(docs)} document(s) loaded from '{data_dir}'")
    return docs

## 3. Split into chunks

Cut documents into pieces of up to 500 characters. Repeat 100 characters across boundaries to retain some context. Each piece keeps a source path and an ID.

In [4]:
# 2. SPLIT: each piece keeps its source path and a number, so we can inspect it later.
@dataclass
class Chunk:
    text: str
    source: str
    chunk_id: int


def _sliding_window(text: str, size: int, overlap: int) -> list[str]:
    # Reject settings that would make empty pieces or prevent us from moving forward.
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require chunk_size > 0 and 0 <= chunk_overlap < chunk_size")
    windows = []

    # With size=500 and overlap=100, start at 0, 400, 800, and so on.
    for start in range(0, len(text), size - overlap):
        windows.append(text[start:start + size])

        # We have covered the end of the text; do not add another repeated tail.
        if start + size >= len(text):
            break
    return windows


def chunk_documents(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Chunk]:
    """Bounded overlapping character windows; preserve each document's source."""

    # Check the settings even if the document list is empty.
    _sliding_window("", chunk_size, chunk_overlap)
    chunks = []
    for doc in docs:
        for text in _sliding_window(doc.text, chunk_size, chunk_overlap):
            if text.strip():

                # Use the current list length as the next ID: 0, 1, 2, ... across all files.
                chunks.append(Chunk(text, doc.source, len(chunks)))
    print(f"[chunk] {len(chunks)} chunk(s) created")
    return chunks

## 4. Create embeddings

MiniLM turns text into 384 numbers. We use the same model for questions and chunks so their vectors can be compared. Normalization makes cosine comparison straightforward.

In [5]:
# 3. EMBED: turn each chunk into numbers. Similar texts can have similar vectors.
# Compute document vectors once, rather than repeat that work for every question.
class Embedder:
    def __init__(self, model_name: str):

        # Import here so loading files and testing the chunker do not need this package.
        from sentence_transformers import SentenceTransformer
        print(f"[embed] loading embedding model '{model_name}' ...")
        self.model = SentenceTransformer(model_name)

    def encode(self, texts: list[str]) -> np.ndarray:
        # Give each vector length one. That lets a dot product measure cosine similarity later.
        vectors = self.model.encode(
            texts,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=len(texts) > 50,
        )

        # Store the numbers in one consistent format.
        return vectors.astype(np.float32)

## 5. Store and search

Keep chunk vectors in a NumPy matrix. Compare the question with every row and return the top three matches. This is exact search on a tiny collection; there is no relevance threshold. Save/load methods are optional.

In [6]:
# 4. STORE AND SEARCH: a NumPy table is enough for this tiny collection.
# Row 0 holds chunk 0's vector, row 1 holds chunk 1's vector, and so on.
class VectorStore:
    def __init__(self, embedder: Embedder):
        self.embedder = embedder
        self.chunks: list[Chunk] = []
        self.matrix: np.ndarray | None = None

    def add(self, chunks: list[Chunk]) -> None:
        if not chunks: raise ValueError("Cannot index empty chunks")
        self.chunks = chunks

        # Keep vector rows in the same order as the chunks, or source labels would be wrong.
        self.matrix = self.embedder.encode([c.text for c in chunks])
        print(f"[index] vector matrix shape: {self.matrix.shape}")

    def search(self, query: str, top_k: int) -> list[tuple[Chunk, float]]:
        assert self.matrix is not None, "Call add() before search()."
        if top_k <= 0 or not query.strip(): raise ValueError("Require a nonempty query and positive top_k")

        # Use the same model for questions and documents so their vectors can be compared.
        q = self.embedder.encode([query])[0]
        # @ compares the question with every stored vector in one operation.
        # Vectors have length one, so these dot products are cosine scores, not probabilities.
        scores = self.matrix @ q
        top_k = min(top_k, len(self.chunks))

        # argsort gives positions from smallest to largest; minus scores puts the best first.
        best = np.argsort(-scores)[:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best]


    # Saving avoids computing document vectors again. SimpleRAG does not call save/load for you.
    def save(self, path: str) -> None:

        # Save ordinary strings, not Python objects that need pickle when loaded.
        np.savez(path, matrix=self.matrix,
                 texts=np.array([c.text for c in self.chunks], dtype=str),
                 sources=np.array([c.source for c in self.chunks], dtype=str))

    def load(self, path: str) -> None:

        # Load only files you expect. You must still use the same embedding model as when saving.
        # This example does not record or verify that model choice in the saved index.
        data = np.load(path, allow_pickle=False)
        self.matrix = data["matrix"]
        self.chunks = [Chunk(t, s, i) for i, (t, s) in enumerate(zip(data["texts"], data["sources"]))]

## 6. Build the prompt

Put numbered passages beside the question and ask the model to answer only from them. This supplies evidence, but does not enforce correct answers or citations.

In [7]:
# 5. PROMPT: ask the writer to use the passages, cite them, and admit missing information.
# These are instructions, not guarantees. Our saved answers omitted the requested inline citations.
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the provided context. "
    "If the answer is not contained in the context, say \"I don't know based on the "
    "provided documents.\" Keep answers concise and cite sources like [1], [2]."
)


def build_messages(question: str, retrieved: list[tuple[Chunk, float]]) -> list[dict]:

    # Label each passage [1], [2], etc. and show its source beside the text.
    context = "\n\n".join(
        f"[{i}] (source: {chunk.source})\n{chunk.text}"
        for i, (chunk, _score) in enumerate(retrieved, start=1)
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"


    # The system message sets instructions; the user message holds evidence and the question.
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

## 7. Load the answer model

Qwen writes the answer locally through Transformers. The optional Ollama class offers another local backend and is not used in the default demonstration. These class definitions do not download models yet.

In [8]:
# 6. GENERATE: load Qwen locally. The first run downloads its files.
class HuggingFaceLLM:
    def __init__(self, model_name: str):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        print(f"[llm] loading LLM '{model_name}' (first run downloads it) ...")
        self.torch = torch
        # The tokenizer converts between readable text and the token IDs the model uses.
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # Let the library choose the saved number format and suitable devices.
        # A larger model may need more memory than the machine has available.
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto",
        )

        # Turn off training-only behavior such as dropout. We are answering, not training.
        self.model.eval()

    def generate(self, messages: list[dict], max_new_tokens: int, temperature: float) -> str:

        # Put the messages into the chat format this particular model expects.
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Convert the prompt into tokens and put them on the model's device.
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)


        # No training gradients are needed, which saves memory and work.
        with self.torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=temperature if temperature > 0 else None,
                top_p=0.9 if temperature > 0 else None,
                pad_token_id=self.tokenizer.eos_token_id,
            )


        # The output contains the original prompt too. Keep only the newly written answer.
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Optional alternative: ask an already-running local Ollama server to generate the answer.
class OllamaLLM:
    """
    Alternative generator using a local Ollama server (https://ollama.com).
    Run `ollama pull llama3.2` once, keep Ollama running, then pass
    --backend ollama --llm_model llama3.2 . Needs only the `requests` package.
    """

    def __init__(self, model_name: str, host: str = "http://localhost:11434"):
        self.model_name = model_name
        self.host = host

    def generate(self, messages: list[dict], max_new_tokens: int, temperature: float) -> str:
        import requests
        resp = requests.post(
            f"{self.host}/api/chat",
            json={
                "model": self.model_name,
                "messages": messages,
                "stream": False,
                "options": {"temperature": temperature, "num_predict": max_new_tokens},
            },
            timeout=300,
        )

        # Stop on an HTTP error instead of pretending the failed request returned an answer.
        resp.raise_for_status()
        return resp.json()["message"]["content"].strip()

## 8. Connect the pipeline

SimpleRAG prepares documents and models once. Its ask method retrieves passages, builds the prompt, generates an answer, and returns the answer together with the evidence.

In [9]:
# 7. CONNECT THE PARTS: prepare the documents and load the models once for this object.
class SimpleRAG:
    def __init__(self, config: RAGConfig, backend: str = "hf"):
        self.config = config


        docs = load_documents(config.data_dir)
        chunks = chunk_documents(docs, config.chunk_size, config.chunk_overlap)
        self.store = VectorStore(Embedder(config.embedding_model))
        self.store.add(chunks)


        self.llm = OllamaLLM(config.llm_model) if backend == "ollama" else HuggingFaceLLM(config.llm_model)

    # Each question follows three steps. No document re-indexing is needed here.
    def ask(self, question: str, verbose: bool = True) -> dict:


        # Retrieve: find the closest document pieces.
        retrieved = self.store.search(question, self.config.top_k)

        # Augment: put those pieces into the prompt alongside the question.
        messages = build_messages(question, retrieved)

        # Generate: let the model write an answer using that prompt.
        answer = self.llm.generate(messages, self.config.max_new_tokens, self.config.temperature)

        if verbose:
            print("\n" + "=" * 70)
            print(f"Q: {question}")
            print("-" * 70)
            print("Retrieved chunks:")
            for i, (chunk, score) in enumerate(retrieved, start=1):
                preview = chunk.text[:100].replace("\n", " ")
                print(f"  [{i}] score={score:.3f}  {chunk.source}: {preview}...")
            print("-" * 70)
            print(f"A: {answer}")
            print("=" * 70)


        # Return evidence separately so a reader can inspect it, even if the answer has no citations.
        # This does not automatically check that every sentence is supported.
        return {
            "question": question,
            "answer": answer,
            "sources": [{"source": c.source, "score": s, "text": c.text, "chunk_id": c.chunk_id} for c, s in retrieved],
        }

## 9. Try the pipeline

First inspect the chunks. Then load the models and ask questions. Model loading and generation can take time on CPU.

In [10]:
# Use the classes defined above. Inspect the files and first chunk before loading models.


config = RAGConfig(temperature=0.0)
docs = load_documents(config.data_dir)
chunks = chunk_documents(docs, config.chunk_size, config.chunk_overlap)
print("Chunks:", len(chunks))
print(chunks[0])


[load] 1 document(s) loaded from 'data'
[chunk] 3 chunk(s) created
Chunks: 3
Chunk(text="# Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a language model prompt, and generates an answer using that context. It can provide information outside a model's training data, but does not guarantee accurate answers.\n\n# This demonstration\n\nThis pipeline uses sentence-transformers/all-MiniLM-L6-v2 as its embedding model. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec", source='rag_notes.md', chunk_id=0)


In [ ]:
# Build the document index and load the answer model. Later questions reuse them.

rag = SimpleRAG(config)


In [ ]:
# Ask a question, then inspect both the answer and the passages used to write it.

result = rag.ask("Which embedding model does this pipeline use?")
print(result["sources"])


In [ ]:
# Check what the model says. A fluent explanation can still add unsupported claims.

rag.ask("What is Retrieval-Augmented Generation and why is it useful?")


In [ ]:
# The documents do not answer this question. See whether the model admits that.

rag.ask("Who won the 1998 FIFA World Cup?")


## 10. Show previously recorded results

For an interview, this lets you inspect the completed five-question run without waiting for inference. **These are saved observations from an earlier run, not new answers generated by this cell.** Run the setup cell first if using a fresh Colab session, then run this cell directly. Each answer is followed by the actual retrieved passages and their similarity scores.

The earlier run did not produce the requested inline citations. There are only three chunks, so top-k=3 returns the entire collection. This demonstrates the pipeline wiring, not measured retrieval accuracy.

In [11]:
import json

recorded = json.loads(Path("results/examples.json").read_text(encoding="utf-8"))
print("Recorded run (UTC):", recorded["executed_at_utc"])
for item in recorded["results"]:
    print("\nQUESTION:", item["question"])
    print("ANSWER:", item["answer"])
    for source in item["sources"]:
        print(f"\n  Source: {source['source']} | chunk {source['chunk_id']} | cosine {source['score']:.3f}")
        print(source["text"])

Recorded run (UTC): 2026-10-19T17:20:51.381786+00:00

QUESTION: Which embedding model does this pipeline use?
ANSWER: The pipeline uses sentence-transformers/all-MiniLM-L6-v2 as its embedding model.

  Source: rag_notes.md | chunk 0 | cosine 0.490
# Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a language model prompt, and generates an answer using that context. It can provide information outside a model's training data, but does not guarantee accurate answers.

# This demonstration

This pipeline uses sentence-transformers/all-MiniLM-L6-v2 as its embedding model. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec

  Source: rag_notes.md | chunk 1 | cosine 0.305
odel. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vectors. Cosine similarity ranks chunks for each question.

The generator is Qwen/Qwen2.5-1.5B-Instruct, 

## What this shows

We connected document loading, chunking, embedding, retrieval, prompt construction and local generation. Source passages are returned for inspection. The system does not automatically verify claims, enforce inline citations, or reject weak retrieval matches. The command-line argument parser stays in `simple_rag.py`; a notebook uses editable settings and question cells instead.

The component tests in `tests/test_simple_rag.py` check the script’s software behavior with model doubles. They are not a benchmark of language-model accuracy.